In [1]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 77.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=dab88668e0e1935f688a339f3bb35805446d81396de7d2c19f1e34842157e07b
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
# ==============================================================================
# SS-VLM: Inference & RAG Generation Engine (Top-Tier Version)
# ==============================================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import timm
import logging
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==============================================================================
# 1. Configuration (修改这里！)
# ==============================================================================
class Config:
    # ⚠️ 请把这里改成你训练好的 .pth 文件路径
    MODEL_PATH = '/kaggle/input/bestvit/keras/default/1/ss_vlm_supcon_best.pth' 
    
    # Data Path
    DATA_DIR = '/kaggle/input/raf-db-dataset/DATASET'
    
    # Model Specs (Must match training)
    MODEL_NAME = 'vit_base_patch16_224'
    NUM_CLASSES = 7
    EMBED_DIM = 768
    PROJ_DIM = 128
    
    # LLM Specs
    LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    
    # RAG Specs
    K_PROTOTYPES = 5     # Prototypes per class
    TOP_K_RETRIEVAL = 9  # Evidence to retrieve
    SEED = 42
    
    # Class Names
    EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']

# Setup Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

# ==============================================================================
# 2. Model Definitions (Required for loading weights)
# ==============================================================================
class SpectralCoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        mid_channels = max(8, in_channels // reduction_ratio)
        self.avg_pool = nn.AvgPool2d(3, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, mid_channels, bias=False),
            nn.GELU(),
            nn.Linear(mid_channels, in_channels, bias=True),
            nn.Sigmoid()
        )
        self.conv_shared = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.GELU()
        )
        self.conv_h = nn.Conv2d(mid_channels, in_channels, 1)
        self.conv_w = nn.Conv2d(mid_channels, in_channels, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        b, c, h, w = x.size()
        low = self.avg_pool(x)
        high = x - low
        w_spec = self.mlp(self.gap(high).view(b, c)).view(b, c, 1, 1)
        x_h = F.adaptive_avg_pool2d(x, (h, 1))
        x_w = F.adaptive_avg_pool2d(x, (1, w))
        cat = torch.cat([x_h, x_w.permute(0, 1, 3, 2)], dim=2)
        f = self.conv_shared(cat)
        f_h, f_w = torch.split(f, [h, w], dim=2)
        a_h = self.sigmoid(self.conv_h(f_h))
        a_w = self.sigmoid(self.conv_w(f_w.permute(0, 1, 3, 2)))
        return identity + identity * (w_spec * a_h * a_w)

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class SS_VLM(nn.Module):
    def __init__(self, cfg=Config):
        super().__init__()
        self.backbone = timm.create_model(cfg.MODEL_NAME, pretrained=False, num_classes=0)
        self.afrn = SpectralCoordinateAttention(in_channels=cfg.EMBED_DIM)
        self.gem = GeMPooling(p=3)
        self.head_norm = nn.LayerNorm(cfg.EMBED_DIM)
        self.projector = nn.Sequential(
            nn.Linear(cfg.EMBED_DIM, cfg.EMBED_DIM),
            nn.GELU(),
            nn.Linear(cfg.EMBED_DIM, cfg.PROJ_DIM)
        )
        self.head = nn.Linear(cfg.EMBED_DIM, cfg.NUM_CLASSES)

    def forward(self, x):
        x = self.backbone.forward_features(x)
        if x.shape[1] == 197: x = x[:, 1:, :] 
        b, n, c = x.shape
        h = w = int(n**0.5)
        x = x.permute(0, 2, 1).view(b, c, h, w)
        x = self.afrn(x)
        x = self.gem(x).flatten(1)
        x = self.head_norm(x)
        logits = self.head(x)
        return logits, x # Return features for RAG

# ==============================================================================
# 3. RAG Inference Engine (with Qwen CoT)
# ==============================================================================
class RAG_Engine:
    def __init__(self, model_path):
        self.model = SS_VLM().to(device)
        
        if os.path.exists(model_path):
            state_dict = torch.load(model_path, map_location=device)
            self.model.load_state_dict(state_dict, strict=True)
            logger.info(f"✅ Weights loaded from {model_path}")
        else:
            raise FileNotFoundError(f"❌ Model not found at {model_path}")
            
        self.model.eval()
        self.bank_vectors = None
        self.bank_labels = None
        
        # Load Qwen
        logger.info(f"🤖 Loading LLM: {Config.LLM_MODEL_ID}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(Config.LLM_MODEL_ID)
            self.llm = AutoModelForCausalLM.from_pretrained(
                Config.LLM_MODEL_ID, 
                torch_dtype=torch.float16, 
                device_map="auto"
            )
        except Exception as e:
            logger.error(f"❌ LLM Load Failed: {e}")
            self.llm = None

    def build_bank(self, loader):
        logger.info("🔨 Building Prototype Memory Bank...")
        feats_all, labels_all = [], []
        
        with torch.no_grad():
            for imgs, labels in tqdm(loader, desc="Extracting Train Feats"):
                imgs = imgs.to(device)
                _, feats = self.model(imgs)
                feats = F.normalize(feats, p=2, dim=1) # Normalize for Cosine Sim
                feats_all.append(feats.cpu().numpy())
                labels_all.append(labels.numpy())
        
        feats_all = np.concatenate(feats_all)
        labels_all = np.concatenate(labels_all)
        
        bank_vecs, bank_lbls = [], []
        for c in range(Config.NUM_CLASSES):
            indices = np.where(labels_all == c)[0]
            if len(indices) == 0: continue
            c_feats = feats_all[indices]
            # KMeans Clustering
            n_clusters = min(Config.K_PROTOTYPES, len(c_feats))
            kmeans = KMeans(n_clusters=n_clusters, random_state=Config.SEED, n_init=10).fit(c_feats)
            for center in kmeans.cluster_centers_:
                bank_vecs.append(center)
                bank_lbls.append(c)
                
        self.bank_vectors = np.array(bank_vecs)
        self.bank_labels = np.array(bank_lbls)
        logger.info(f"✅ Bank Built. Total Prototypes: {len(bank_vecs)}")

    def generate_clinical_report(self, pred_idx, conf, top_k_indices, gt_idx):
        if self.llm is None: return "LLM Not Available"
        
        pred_emotion = Config.EMOTIONS[pred_idx]
        gt_emotion = Config.EMOTIONS[gt_idx]
        
        # 1. Summarize Evidence
        retrieved_labels = self.bank_labels[top_k_indices]
        retrieved_emotions = [Config.EMOTIONS[l] for l in retrieved_labels]
        unique, counts = np.unique(retrieved_emotions, return_counts=True)
        evidence_summary = "; ".join([f"{count} case(s) of {emo}" for emo, count in zip(unique, counts)])
        
        # 2. Structured Chain-of-Thought (CoT) Prompt
        # This structure forces Qwen to think before diagnosing
        messages = [
            {"role": "system", "content": "You are an expert AI clinician specializing in facial micro-expression diagnostics. Be precise, professional, and evidence-based."},
            {"role": "user", "content": f"""
            Generate a diagnostic report based on the following multi-modal data:
            
            [Visual Sensor Data]
            - Predicted Class: {pred_emotion}
            - Confidence Score: {conf:.2f}
            
            [Historical Knowledge Base (Retrieval)]
            - Retrieved Prototypes: {evidence_summary}
            
            [Instruction]
            Perform a structured analysis using the following steps:
            1. **Visual Analysis**: Interpret the confidence and prediction.
            2. **Evidence Synthesis**: Check if the retrieved historical cases support the visual prediction (Consistency Check).
            3. **Final Diagnosis**: Conclude the emotion.
            
            Output strictly in this format:
            **Visual Analysis**: ...
            **Evidence Synthesis**: ...
            **Final Diagnosis**: ...
            """}
        ]
        
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([text], return_tensors="pt").to(device)
        
        with torch.no_grad():
            generated_ids = self.llm.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.6,    # Slightly lower temp for more stable reasoning
                top_p=0.9,
                do_sample=True
            )
            
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
        ]
        return self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    def run_inference(self, test_loader):
        if self.bank_vectors is None: raise ValueError("Build Memory Bank first!")
        
        logger.info("🔍 Running RAG Inference on Test Set...")
        agreement_count = 0
        purity_sum = 0
        total = 0
        
        case_studies = [] # Store examples for paper
        
        with torch.no_grad():
            for i, (imgs, labels) in enumerate(tqdm(test_loader, desc="Inference")):
                imgs = imgs.to(device)
                
                # Forward
                logits, query_feats = self.model(imgs)
                query_feats = F.normalize(query_feats, p=2, dim=1).cpu().numpy()
                probs = torch.softmax(logits, dim=1)
                confs, preds = torch.max(probs, 1)
                
                # Retrieval
                sims = cosine_similarity(query_feats, self.bank_vectors)
                
                for j in range(len(imgs)):
                    # Top-K
                    top_k_idx = np.argsort(sims[j])[::-1][:Config.TOP_K_RETRIEVAL]
                    ret_labels = self.bank_labels[top_k_idx]
                    
                    # Metrics
                    pred_lbl = preds[j].item()
                    
                    # Agreement (Majority Vote)
                    if np.argmax(np.bincount(ret_labels, minlength=7)) == pred_lbl:
                        agreement_count += 1
                    
                    # Purity
                    purity_sum += np.sum(ret_labels == pred_lbl) / Config.TOP_K_RETRIEVAL
                    total += 1
                    
                    # Generate Reports for the first 5 samples
                    if len(case_studies) < 5:
                        report = self.generate_clinical_report(pred_lbl, confs[j].item(), top_k_idx, labels[j].item())
                        case_studies.append({
                            "gt": Config.EMOTIONS[labels[j].item()],
                            "pred": Config.EMOTIONS[pred_lbl],
                            "report": report
                        })
        
        # Final Stats
        agreement = agreement_count / total * 100
        purity = purity_sum / total * 100
        
        print(f"\n📊 === FINAL RESULTS ===")
        print(f"Agreement@5: {agreement:.2f}%")
        print(f"Purity@5:    {purity:.2f}%")
        
        print(f"\n📝 === Qwen2.5 Generated Reports (CoT) ===")
        for case in case_studies:
            print(f"\n[Ground Truth: {case['gt']} | Prediction: {case['pred']}]")
            print(case['report'])
            print("-" * 60)

# ==============================================================================
# 4. Main Execution
# ==============================================================================
if __name__ == '__main__':
    seed_everything(Config.SEED)
    
    # 1. Load Data
    val_tfm = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    
    try:
        root = Config.DATA_DIR
        train_path = os.path.join(root, 'train')
        if not os.path.exists(train_path): train_path = os.path.join(root, 'original/train')
        test_path = train_path.replace('train', 'test')
        
        # Batch size for inference can be larger
        train_ds = datasets.ImageFolder(train_path, transform=val_tfm)
        test_ds = datasets.ImageFolder(test_path, transform=val_tfm)
        
        train_dl = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2)
        test_dl = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
        logger.info("✅ Data Loaded Successfully")
        
        # 2. Run Pipeline
        rag = RAG_Engine(Config.MODEL_PATH)
        rag.build_bank(train_dl) # Extract features from train set
        rag.run_inference(test_dl) # Evaluate on test set
        
    except Exception as e:
        logger.error(f"❌ Execution failed: {e}")

In [ ]:
# ==============================================================================
# RAG Inference Engine - Baseline Compatibility Version
# ==============================================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import timm
import logging
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==============================================================================
# 1. Configuration (Baseline 设置)
# ==============================================================================
class Config:
    # ⚠️ 把这里换成你的 ViT Baseline 权重路径
    MODEL_PATH = '/kaggle/input/testforijcnn/keras/default/1/vit_baseline_seed44.pth' # <--- 修改这里
    
    # ⚠️ 开启 Baseline 模式 (这很重要！)
    IS_BASELINE = True  
    
    DATA_DIR = '/kaggle/input/raf-db-dataset/DATASET'
    MODEL_NAME = 'vit_base_patch16_224'
    NUM_CLASSES = 7
    LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    K_PROTOTYPES = 5
    TOP_K_RETRIEVAL = 9
    SEED = 42
    EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

# ==============================================================================
# 2. Model Definitions (包含 Baseline 和 SS-VLM)
# ==============================================================================

# --- A. 你的 SS-VLM 组件 (保持不变) ---
class SpectralCoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        mid_channels = max(8, in_channels // reduction_ratio)
        self.avg_pool = nn.AvgPool2d(3, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(nn.Linear(in_channels, mid_channels, bias=False), nn.GELU(), nn.Linear(mid_channels, in_channels, bias=True), nn.Sigmoid())
        self.conv_shared = nn.Sequential(nn.Conv2d(in_channels, mid_channels, 1, bias=False), nn.BatchNorm2d(mid_channels), nn.GELU())
        self.conv_h = nn.Conv2d(mid_channels, in_channels, 1)
        self.conv_w = nn.Conv2d(mid_channels, in_channels, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        identity = x; b, c, h, w = x.size()
        low = self.avg_pool(x); high = x - low
        w_spec = self.mlp(self.gap(high).view(b, c)).view(b, c, 1, 1)
        x_h = F.adaptive_avg_pool2d(x, (h, 1)); x_w = F.adaptive_avg_pool2d(x, (1, w))
        cat = torch.cat([x_h, x_w.permute(0, 1, 3, 2)], dim=2)
        f = self.conv_shared(cat); f_h, f_w = torch.split(f, [h, w], dim=2)
        return identity + identity * (w_spec * self.sigmoid(self.conv_h(f_h)) * self.sigmoid(self.conv_w(f_w.permute(0, 1, 3, 2))))

class GeMPooling(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__(); self.p = nn.Parameter(torch.ones(1) * p); self.eps = eps
    def forward(self, x): return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class SS_VLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(Config.MODEL_NAME, pretrained=False, num_classes=0)
        self.afrn = SpectralCoordinateAttention(768)
        self.gem = GeMPooling(p=3)
        self.head_norm = nn.LayerNorm(768)
        self.projector = nn.Sequential(nn.Linear(768, 768), nn.GELU(), nn.Linear(768, 128))
        self.head = nn.Linear(768, Config.NUM_CLASSES)
    def forward(self, x):
        x = self.backbone.forward_features(x)
        if x.shape[1] == 197: x = x[:, 1:, :]
        b, n, c = x.shape; h = w = int(n**0.5)
        x = x.permute(0, 2, 1).view(b, c, h, w)
        x = self.afrn(x); x = self.gem(x).flatten(1); x = self.head_norm(x)
        return self.head(x), x

# --- B. 简易 Baseline 模型 (专门用来加载你的 Baseline pth) ---
class ViT_Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(Config.MODEL_NAME, pretrained=False, num_classes=0)
        
        # [Fix 1] 还原 MLP Head 结构
        # 你的报错显示有 head.0 和 head.3，说明是 Sequential(Linear, Act, Drop, Linear)
        self.head = nn.Sequential(
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, Config.NUM_CLASSES)
        )
        # ViT 通常不需要额外的 norm，因为 backbone 输出前已经 norm 了
        # 但为了防止权重里的 norm 没地方放，我们定义一个 dummy
        self.norm = nn.Identity() 

    def forward(self, x):
        x = self.backbone.forward_features(x)
        # 处理 ViT 输出: [B, 197, 768] -> CLS Token [B, 768]
        if x.shape[1] == 197:
            feat = x[:, 0, :]
        else:
            feat = x.mean(dim=1)
            
        logits = self.head(feat)
        return logits, feat

# ==============================================================================
# 3. RAG Engine (自动适配版)
# ==============================================================================
class RAG_Engine:
    def __init__(self, model_path, is_baseline=False):
        if is_baseline:
            logger.info("⚠️ Mode: ViT Baseline (Compatibility Mode)")
            self.model = ViT_Baseline().to(device)
        else:
            logger.info("🚀 Mode: SS-VLM (Ours)")
            self.model = SS_VLM().to(device)
        
        if os.path.exists(model_path):
            try:
                ckpt = torch.load(model_path, map_location=device)
                # 自动解包
                state_dict = ckpt['model'] if (isinstance(ckpt, dict) and 'model' in ckpt) else ckpt
                
                # [Fix 2] 智能 Key 映射 (Auto-Remapping)
                new_state_dict = {}
                for k, v in state_dict.items():
                    name = k
                    
                    # 1. 去掉 DDP 的 module. 前缀
                    if name.startswith('module.'): 
                        name = name[7:]
                        
                    # 2. 关键修复：把 'vit.' 替换成 'backbone.'
                    if name.startswith('vit.'):
                        name = name.replace('vit.', 'backbone.')
                        
                    new_state_dict[name] = v
                
                # 加载权重
                # strict=False 允许忽略一些无关紧要的 missing keys (比如 pos_embed resize 问题)
                missing, unexpected = self.model.load_state_dict(new_state_dict, strict=False)
                
                logger.info(f"✅ Weights Loaded.")
                if len(missing) > 0: logger.warning(f"Note - Missing keys: {len(missing)}")
                
            except Exception as e:
                logger.error(f"❌ Load Failed: {e}")
                raise e
        else:
            raise FileNotFoundError(f"not found: {model_path}")
            
        self.model.eval()
        self.bank_vectors = None
        self.bank_labels = None
        
        # Load LLM
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(Config.LLM_MODEL_ID)
            self.llm = AutoModelForCausalLM.from_pretrained(Config.LLM_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
        except:
            self.llm = None

    # ... (build_bank, generate_report, run 方法不需要变，直接用之前的即可) ...
    # 为了方便你直接运行，我把剩下的方法也补全在这里
    
    def build_bank(self, loader):
        logger.info("🔨 Building Bank...")
        feats, lbls = [], []
        with torch.no_grad():
            for x, y in tqdm(loader):
                _, f = self.model(x.to(device))
                feats.append(F.normalize(f, p=2, dim=1).cpu().numpy())
                lbls.append(y.numpy())
        feats = np.concatenate(feats); lbls = np.concatenate(lbls)
        
        vecs, blbls = [], []
        for c in range(Config.NUM_CLASSES):
            idx = np.where(lbls == c)[0]
            if len(idx) == 0: continue
            kmeans = KMeans(n_clusters=min(Config.K_PROTOTYPES, len(idx)), random_state=42).fit(feats[idx])
            for center in kmeans.cluster_centers_:
                vecs.append(center); blbls.append(c)
        self.bank_vectors = np.array(vecs); self.bank_labels = np.array(blbls)

    def generate_report(self, pred, conf, top_k, gt):
        if self.llm is None: return "LLM NA"
        
        # [修改这里] 加上 "similar historical cases of"
        evid = "; ".join([f"{count} similar historical cases of {Config.EMOTIONS[l]}" for l, count in zip(*np.unique(self.bank_labels[top_k], return_counts=True))])
        
        # [修改这里] Prompt 稍微强硬一点，防止它胡言乱语
        msg = [
            {"role": "system", "content": "You are an expert clinician. Write a short diagnostic report."},
            {"role": "user", "content": f"Prediction: {Config.EMOTIONS[pred]} (Conf: {conf:.2f}). Retrieval Evidence: {evid}. Task: Synthesize the evidence to support or question the diagnosis."}
        ]
        
        txt = self.tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer([txt], return_tensors="pt").to(device)
        with torch.no_grad():
            out = self.llm.generate(**inputs, max_new_tokens=100, temperature=0.6) # 温度调低点
        return self.tokenizer.decode(out[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    def run(self, loader):
        acc, pur, total = 0, 0, 0
        case_studies = []
        with torch.no_grad():
            for x, y in tqdm(loader):
                x = x.to(device)
                logits, qf = self.model(x)
                sims = cosine_similarity(F.normalize(qf, p=2, dim=1).cpu().numpy(), self.bank_vectors)
                confs, preds = torch.max(torch.softmax(logits, dim=1), 1)
                
                for j in range(len(x)):
                    topk = np.argsort(sims[j])[::-1][:Config.TOP_K_RETRIEVAL]
                    ret = self.bank_labels[topk]
                    if np.argmax(np.bincount(ret, minlength=7)) == preds[j]: acc += 1
                    pur += np.sum(ret == preds[j].item()) / 5.0
                    total += 1
                    if len(case_studies) < 5:
                        case_studies.append((Config.EMOTIONS[y[j]], Config.EMOTIONS[preds[j]], self.generate_report(preds[j], confs[j], topk, y[j])))
        
        print(f"Metrics -> Agr: {acc/total:.2%}, Pur: {pur/total:.2%}")
        for gt, pr, rep in case_studies: print(f"[GT:{gt}|Pred:{pr}]\n{rep}\n{'-'*40}")

# ==============================================================================
# 4. Main
# ==============================================================================
if __name__ == '__main__':
    seed_everything(Config.SEED)
    
    # Load Data
    val_tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
    try:
        tr_ds = datasets.ImageFolder(os.path.join(Config.DATA_DIR, 'train'), transform=val_tfm)
        te_ds = datasets.ImageFolder(os.path.join(Config.DATA_DIR, 'test'), transform=val_tfm)
    except:
        tr_ds = datasets.ImageFolder(os.path.join(Config.DATA_DIR, 'original/train'), transform=val_tfm)
        te_ds = datasets.ImageFolder(os.path.join(Config.DATA_DIR, 'original/test'), transform=val_tfm)

    tr_dl = DataLoader(tr_ds, batch_size=64, shuffle=False)
    te_dl = DataLoader(te_ds, batch_size=64, shuffle=False)

    # Run
    rag = RAG_Engine(Config.MODEL_PATH, is_baseline=Config.IS_BASELINE)
    rag.build_bank(tr_dl)
    rag.run(te_dl)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

2026-01-25 12:51:21,284 - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

2026-01-25 12:51:22,731 - 🔨 Building Bank...
100%|██████████| 48/48 [01:07<00:00,  1.42s/it]

Metrics -> Agr: 99.22%, Pur: 98.47%
[GT:Surprise|Pred:Surprise]
**Diagnostic Report: Surprise**

**Patient Characteristics:** 
- Age: [Age]
- Gender: [Gender]
- Symptom History: [Brief summary of patient's medical history and current symptoms]

**Clinical Presentation:**
- Presenting Symptoms: [List all observed symptoms]
- Physical Examination Findings: [Describe any physical findings, including vital signs if relevant]
- Diagnostic Tests: [List any tests performed and results]

**Historical Context:**
- Medical Record Review: [Summar
----------------------------------------
[GT:Surprise|Pred:Surprise]
**Diagnostic Report**

**Patient Information:**  
- Age: [Age]
- Gender: [Gender]
- Medical History: [Brief medical history]

**Diagnosis:**  
Surprise Syndrome

**Evidence Synthesis:**
The patient presents with symptoms consistent with Surprise Syndrome, as evidenced by the following:

1. **Symptoms:** The patient exhibits sudden onset of severe headache, nausea, and visual disturbance